In [ ]:
import requests

url = ('https://newsapi.org/v2/everything?'
       'q=Apple&'
       'from=2025-04-20&'
       'sortBy=popularity&'
       'apiKey=aa97db5ece6a4d2093fceadcc9f0bf92')

response = requests.get(url)
# print response.json

In [ ]:
from datetime import timedelta
import datetime
from os import environ
try:
    from newsapi.newsapi_client import NewsApiClient
except:
    !pip3 install newsapi.newsapi_client

newsapi = NewsApiClient(api_key='ABC')
# to_date = datetime.date.today() - timedelta(days = 1)
to_date = to_date = datetime.date(2024, 10, 31) - timedelta(days = 1)
from_date = to_date - datetime.timedelta(days=28)

def filter_articles(articles: dict):
    filter_n_articles = articles['articles'][0:len(articles['articles'])]
    list_of_articles = []
    for article in filter_n_articles:
        if "\'" not in article['description'] and 'www.macrumors.com' not in article['url']: 
            a = {
                'source_name': article['source']['name'],
                'author': article['author'],
                'title': article['title'],
                'description': article['description'],
                'published': article['publishedAt'],
                'url': article['url']
            }
        else: continue
        list_of_articles.append(a)
    return list_of_articles


all_articles = newsapi.get_everything(q='apple',
                                       from_param = from_date,
                                       to = to_date,
                                       language = 'en',
                                       sort_by = 'relevancy')
filter_articles(all_articles)

NewsAPIException: {'status': 'error', 'code': 'parameterInvalid', 'message': 'You are trying to request results too far in the past. Your plan permits you to request articles as far back as 2025-04-27, but you have requested 2024-10-02. You may need to upgrade to a paid plan.'}

In [7]:
all_articles = newsapi.get_everything(q='apple',
                                       from_param = from_date,
                                       to = to_date,
                                       language = 'en',
                                       sort_by = 'relevancy')
print(all_articles)
articles_list = []
filter_n_articles = all_articles['articles'][0:len(all_articles['articles'])]
for article in filter_n_articles:
    if "\'" not in article['description'] and 'www.macrumors.com' not in article['url']:
        articles_list.append(article)

print(len(articles_list))

{'status': 'ok', 'totalResults': 21860, 'articles': [{'source': {'id': 'the-verge', 'name': 'The Verge'}, 'author': 'Umar Shakir', 'title': 'Apple has a new ‘Viral’ playlist on Apple Music and Shazam', 'description': 'Apple is launching a new global Viral Chart playlist in Apple Music that consists of tracks people are discovering through the company’s Shazam service. The playlist, which is updated daily, shows the top 50 songs people have heard playing out in the real wor…', 'url': 'https://www.theverge.com/news/663866/apple-music-viral-charts-global-playlist-shazam-top-50', 'urlToImage': 'https://platform.theverge.com/wp-content/uploads/sites/2/chorus/uploads/chorus_asset/file/23925979/acastro_STK046_03.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200', 'publishedAt': '2025-05-08T21:01:29Z', 'content': 'Now Apple and Shazam users have a quick place to see what music people are discovering in the real world.\r\nNow Apple and Shazam users have a quick pl

In [10]:
import pandas as pd

filtered_articles = filter_articles(all_articles)
df_articles = pd.DataFrame(filtered_articles)
df_articles.to_excel('apple_articles.xlsx', index=False)

print("Saved!")

Saved!


In [119]:
import asyncio
from crawl4ai import *

async def extract_Article_body(url, baseSelector, selector):
    browser_config = BrowserConfig(
        browser_type="chromium",  # Options: "chromium", "firefox", "webkit"
        headless=True,            # Set to False if you want to see the browser window
        viewport_width=1280,
        viewport_height=800,
        verbose=True,
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
    )
    async with AsyncWebCrawler(config = browser_config) as crawler:

        json_strategy = JsonCssExtractionStrategy(
        schema = {
            "name": "Article Body",
            "baseSelector": baseSelector,
            "fields" : [
                {
                "name": "Section",
                "selector": selector,
                "type": "text"
                }
            ]
                },
        verbose = False
        )

        crawler_config = CrawlerRunConfig(
            extraction_strategy = json_strategy,
            screenshot=False,
            verbose=True,
            cache_mode=CacheMode.DISABLED,
            log_console=True,
            # wait_for=".swiper-slide-body",
            scan_full_page=True,  # Tells the crawler to try scrolling the entire page
            scroll_delay=2,     # Delay (seconds) between scroll steps
        )

        result = await crawler.arun(
            url = url, 
            config = crawler_config
        )
        # print(result.markdown)
        return result.extracted_content

In [120]:
async def process_each_article(verge_articles):
    extracted_content_list = []
    for each_url in verge_articles["url"]:
        extracted_content = await extract_Article_body(each_url, "#content > article > div.duet--layout--entry-body-container._1t5ltw90._1ymtmqp3._1ymtmqp13", "#zephr-anchor")
        # if not extracted_content:
            # extracted_content = await extract_Article_body(each_url, "#content > article > div.duet--layout--entry-body-container._1t5ltw90._1ymtmqp3._1ymtmqp13 > div", "#zephr-anchor")
        extracted_content_list.append(extracted_content)
    return extracted_content_list

if __name__ == "__main__":
    df_articles = pd.DataFrame(filtered_articles)
    verge_articles = df_articles[df_articles["source_name"] == "The Verge"]
    extracted_content_list = await process_each_article(verge_articles)
    print(extracted_content_list)


[INIT].... → Crawl4AI 0.6.3 

[CONSOLE]. ℹ Console: player_name=VERGE_DESK_HIGH_RR_SS

[CONSOLE]. ℹ Console: Failed to load resource: net::ERR_SOCKET_NOT_CONNECTED

[CONSOLE]. ℹ Console: Refused to execute script from 
'https://x.bidswitch.net/sync?ssp=connatix&user_id=2-24f5aedd84a74985a89f396c186c90fd&gdpr=0' because its MIME type
('image/gif') is not executable.

[CONSOLE]. ℹ Console: Syncing cannot be made with  
https://csync.loopme.me/?redirect=https%3A%2F%2Fcks.connatix.com%2Fcks%3Fpid%3D18%26ev%3D2-24f5aedd84a74985a89f396c
186c90fd%26pname%3DLoopMe%26api-tier%3D1%26uid%3D%7Bdevice_id%7D%26pubid%3D11186&gdpr=0

...

[FETCH]... ↓ https://www.theverge.com/news/661047/eric-migico...e-core-devices-core-2-duo-core-time-2-smartwatch  |
✓ | ⏱: 18.74s 

[SCRAPE].. ◆ https://www.theverge.com/news/661047/eric-migico...e-core-devices-core-2-duo-core-time-2-smartwatch  |
✓ | ⏱: 0.10s 

[EXTRACT]. ■ Completed for https://www.theverge.com/news/661047/eric-migicovs... | Time: 0.022023125000487198s 

[COMPLETE] ● https://www.theverge.com/news/661047/eric-migico...e-core-devices-core-2-duo-core-time-2-smartwatch  |
✓ | ⏱: 18.88s 

['[\n    {\n        "Section": "Apple is launching a new globalViral Chart playlist in Apple Musicthat consists of tracks people are discovering through the company’s Shazam service. The playlist, which is updated daily, shows the top 50 songs people have heard playing out in the real world and have logged through Shazam. You can see chartson Shazam’s website as well.The playlists use “Shazam’s data to offer a comprehensive view of today’s fastest-growing songs across the globe, David Emery, who works for Apple Music in the UK,says on Threads. Emery notes that charts reflect songs going viral on Shazam in “real time” and then ranks them based on “their weekly growth in Shazam volume.”As of writing, the top viral songs in the list includeShake It To The Max (FLY) [Remix]by Moliy,Nothing’s Gonna Stop Us Nowby Starship, andHot Togetherby The Pointer Sisters. On Shazam’s website, you can look at the rankings by country, if you’d like.4Comments/4NewSee More:AppleEntertainmentMusicNewsTech"\

In [ ]:
import json
from json import loads
# verge_articles = df_articles[df_articles["source_name"] == "The Verge"]
# verge_articles_copied.insert(6, "article_body", None)
verge_articles = verge_articles.reset_index(drop=True)
verge_articles_copied = verge_articles.copy()
extracted_content_listV2 = extracted_content_list.copy()
verge_articles_copied["article_body"] = "Empty"
for i in range(len(extracted_content_listV2)):
    json_article_content = loads(extracted_content_listV2[i])
    # verge_articles_copied["article_body"].loc[i] = json_article_content[0]["Section"]
    # print(i)
    if len(json_article_content) != 0:
        # print(i)
        verge_articles_copied.at[i, "article_body"] = json_article_content[0]["Section"]
    else:
        print("empty " + str(i))
        verge_articles_copied.at[i, "article_body"] = "Empty"

    # verge_articles_copied["article_body"].loc[i] = extracted_content_listV2[i][extracted_content_listV2[i].find(":"):-7]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
empty 19
20
21
22
23
24
25
26
27
28
29
30
31
32
empty 33
34
35
36
37


In [151]:
def split_text_into_chunks(text, max_words):
    words = text.split()
    return [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

max_words = 5000

new_rows = []
for _, row in verge_articles_copied.iterrows():
    chunks = split_text_into_chunks(row['article_body'], max_words)
    new_row = row.to_dict()
    for i, chunk in enumerate(chunks):
        if i == 0:
            new_row['article_body'] = chunk
        else:
           new_row[f'article_body_part_{i}'] = chunk
    new_rows.append(new_row)

df_expanded = pd.DataFrame(new_rows)

# #Save to Excel
df_expanded.to_excel('apple_Verge_articles_with_body_2.xlsx', index=False)

print("Saved!")

Saved!


In [ ]:
# extracted_content_listV2 = extracted_content_list.copy()
# # print(extracted_content_listV2[37])

# print(type(extracted_content_list))
# for i,l in enumerate(extracted_content_list):
#     if i > 17:
#         print(l)
#     # print(str(i) + " " + str(type(l)))

# print(len(extracted_content_list))
# import json
# from json import loads
# json_str = loads(extracted_content_list[19])
# print(json_str)
# # print(json_str[0]["Section"])
# Ignore this code block